Wojciech Kucharski 240435

Bartłomiej Art

In [7]:
!pip install pulp

In [8]:
import pulp
from pulp import *
from ipywidgets import interact, interactive, fixed, interact_manual, Layout, FloatSlider, IntSlider
import ipywidgets as widgets

In [9]:
prob = LpProblem("Produkcja problem",LpMaximize)
sztukA=LpVariable("sztukA",0,None,LpInteger)
sztukB=LpVariable("sztukB",0,None,LpInteger)
sztukC=LpVariable("sztukC",0,None,LpInteger)


prob += 400*sztukA + 300*sztukB +200*sztukC, "Optymalizacja zysku"
prob += 0.3*sztukA + 0.5*sztukB+0.4*sztukC <= 1800, "Montaż godzin"
prob += 0.1*sztukA + 0.08*sztukB+0.04*sztukC <= 800, "Kontrola godzin"
prob += 0.06*sztukA + 0.04*sztukB+0.05*sztukC <= 700, "Pakowanie godzin"
prob.writeLP("Kontrola.lp")
prob.solve()

for v in prob.variables():
    print(v.name, "=", v.varValue)
print("Całkowity zysk = ", value(prob.objective))

sztukA = 6000.0
sztukB = 0.0
sztukC = 0.0
Całkowity zysk =  2400000.0


In [10]:
import matplotlib.pyplot as plt
from pulp import LpVariable, LpProblem, LpMaximize, lpSum, LpStatus, value
from ipywidgets import interact, FloatSlider, Layout

Ingredients = ["CZEKOLADA", "KARMEL", "CUKIER", "ORZECHY", "OLEJ", "OPAKOWANIA"]

mars = {
    "CZEKOLADA": 0.30,
    "KARMEL": 0.40,
    "CUKIER": 0.33,
    "ORZECHY": 0.07,
    "OLEJ": 0.06,
    "OPAKOWANIA": 1
}

snickers = {
    "CZEKOLADA": 0.45,
    "KARMEL": 0.11,
    "CUKIER": 0.22,
    "ORZECHY": 0.51,
    "OLEJ": 0.07,
    "OPAKOWANIA": 1
}

Price = {
    "CZEKOLADA": 22,
    "KARMEL": 16,
    "ORZECHY": 21,
    "CUKIER": 11,
    "OLEJ": 8,
    "OPAKOWANIA": 2
}


style = {'description_width': 'initial'}
czekoladaMax_slider = FloatSlider(min=0, max=15000, value=8000,
                                  description="ograniczenieCzekolady",
                                  style=style, layout=Layout(width='400px'))

orzechyMax_slider = FloatSlider(min=0, max=4000, value=2000,
                                description="ograniczenieOrzechów",
                                style=style, layout=Layout(width='400px'))

opakowanieMax_slider = FloatSlider(min=0, max=25000, value=12000,
                                   description="ograniczenieOpakowań",
                                   style=style, layout=Layout(width='400px'))

marsPrice_slider = FloatSlider(min=85, max=115, value=95,
                               description="cenaMarsów",
                               style=style, layout=Layout(width='400px'))

snickersPrice_slider = FloatSlider(min=85, max=115, value=110,
                                   description="cenaSnickersów",
                                   style=style, layout=Layout(width='400px'))


def optimizeProduction(czekoladaMax=8000, orzechyMax=2000, opakowanieMax=12000,
                       marsPrice=95, snickersPrice=110):

    model = LpProblem("The money problem", LpMaximize)
    marsAmount = LpVariable('marsAmount', lowBound=0, cat='LPInteger')
    snickersAmount = LpVariable('snickersAmount', lowBound=0, cat='LPInteger')
    marsCost, snickersCost = 0, 0

    for ingredient in Ingredients:
        marsCost += mars[ingredient] * Price[ingredient]
        snickersCost += snickers[ingredient] * Price[ingredient]

    model += ((marsPrice - marsCost) * marsAmount
              + (snickersPrice - snickersCost) * snickersAmount), "Maximize_Profit"
    model += marsAmount * mars["ORZECHY"] + snickersAmount * snickers["ORZECHY"] <= orzechyMax, "orzechyMax"
    model += marsAmount * mars["CZEKOLADA"] + snickersAmount * snickers["CZEKOLADA"] <= czekoladaMax, "czekoladaMax"
    model += marsAmount * mars["OPAKOWANIA"] + snickersAmount * snickers["OPAKOWANIA"] <= opakowanieMax, "opakowanieMax"

    model.solve()

    print("=== Profit per bar ===")
    print("Mars profit per bar:", marsPrice - marsCost)
    print("Snickers profit per bar:", snickersPrice - snickersCost)
    print("")

    print("=== Leftover Resources ===")
    leftover_choc = czekoladaMax - (marsAmount.varValue * mars["CZEKOLADA"]
                                    + snickersAmount.varValue * snickers["CZEKOLADA"])
    leftover_nuts = orzechyMax - (marsAmount.varValue * mars["ORZECHY"]
                                  + snickersAmount.varValue * snickers["ORZECHY"])
    leftover_wrap = opakowanieMax - (marsAmount.varValue * mars["OPAKOWANIA"]
                                     + snickersAmount.varValue * snickers["OPAKOWANIA"])

    print("Czekolada left:", leftover_choc)
    print("Orzechy left:", leftover_nuts)
    print("Opakowania left:", leftover_wrap)
    print("")

    print("=== Solution Status ===")
    print("Status of the solver:", LpStatus[model.status])
    print("")

    print("=== Decision Variables ===")
    for var in model.variables():
        print(f"{var.name} =", var.varValue)
    print("")

    final_profit = value(model.objective)
    print("=== Total Profit ===")
    print("Total profit result =", final_profit)

    fig, ax = plt.subplots()
    ax.bar(["Mars", "Snickers"], [marsAmount.varValue, snickersAmount.varValue],
           color=["#FFA07A", "#8A2BE2"])
    ax.set_title("Optimal Production Plan")
    ax.set_ylabel("Quantity Produced")
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

interact(
    optimizeProduction,
    czekoladaMax=czekoladaMax_slider,
    orzechyMax=orzechyMax_slider,
    opakowanieMax=opakowanieMax_slider,
    marsPrice=marsPrice_slider,
    snickersPrice=snickersPrice_slider
)


interactive(children=(FloatSlider(value=8000.0, description='ograniczenieCzekolady', layout=Layout(width='400p…

<function __main__.optimizeProduction(czekoladaMax=8000, orzechyMax=2000, opakowanieMax=12000, marsPrice=95, snickersPrice=110)>